# Forest Ecoregion Clustering — Experiment
**Task-driven Weighted Soft K-Means vs KMeans baseline**

По мотивам: Kharitonova et al. (2025), Doklady Earth Sciences, DOI: 10.1134/S1028334X24604346

In [ ]:
# ── Шаг 0: Установка и клонирование ──────────────────────────────────────────
import subprocess, sys

REPO = 'https://github.com/Alexeiyaganov/ForestEcoregionClustering.git'
BRANCH = 'main'

!git clone --branch {BRANCH} {REPO} repo 2>&1 | tail -3
%cd repo
!pip install -q pyyaml requests scikit-learn matplotlib pandas numpy

In [ ]:
# ── Шаг 1: Загрузка конфига ──────────────────────────────────────────────────
import sys; sys.path.insert(0, '.')
import yaml

with open('configs/config.yaml') as f:
    config = yaml.safe_load(f)

print('Эксперимент:', config['experiment']['name'])
print('Регион: lat', config['region']['lat_min'], '–', config['region']['lat_max'],
      '| lon', config['region']['lon_min'], '–', config['region']['lon_max'])
print('Признаки:', config['data']['feature_cols'])

In [ ]:
# ── Шаг 2: Данные ────────────────────────────────────────────────────────────
from data.pipeline import build_dataset

df = build_dataset(config)
df.head()

In [ ]:
# ── Шаг 3: Baseline (KMeans) ─────────────────────────────────────────────────
from models.baseline import fit_baseline

print('\n=== BASELINE: KMeans ===')
baseline = fit_baseline(df, config)
df['cluster_baseline'] = baseline.labels

In [ ]:
# ── Шаг 4: Weighted clustering (наш метод) ───────────────────────────────────
from models.weighted_clustering import fit_weighted

print('\n=== WEIGHTED SOFT K-MEANS ===')
weighted = fit_weighted(df, config)
df['cluster_weighted'] = weighted.labels

In [ ]:
# ── Шаг 5: Метрики ───────────────────────────────────────────────────────────
from eval.metrics import compute_all_metrics, compare_models

metrics_baseline = compute_all_metrics(
    df, baseline.labels, baseline.X_scaled, config, 'KMeans baseline'
)
metrics_weighted = compute_all_metrics(
    df, weighted.labels, weighted.X_scaled, config, 'Weighted clustering'
)

df_compare = compare_models([metrics_baseline, metrics_weighted])
print('\n' + '='*60)
print('ИТОГОВОЕ СРАВНЕНИЕ:')
print(df_compare.to_string())

In [ ]:
# ── Шаг 6: Визуализация ──────────────────────────────────────────────────────
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from viz.plots import (
    plot_comparison_maps,
    plot_feature_weights,
    plot_training_curve,
    plot_metrics_table,
    plot_nee_by_cluster,
)
import numpy as np

plot_comparison_maps(df, baseline.labels, weighted.labels)
plot_feature_weights(
    config['data']['feature_cols'],
    baseline.weights,
    weighted.weights,
)
plot_training_curve(
    weighted.loss_history,
    weighted.weight_history,
    config['data']['feature_cols'],
)
plot_metrics_table(df_compare)
plot_nee_by_cluster(
    df, baseline.labels, weighted.labels,
    target_col=config['data']['target_col'],
)

In [ ]:
# ── Шаг 7: Сохранение результатов ────────────────────────────────────────────
import json
from pathlib import Path

Path('results').mkdir(exist_ok=True)

# Таблица сравнения
df_compare.to_csv('results/comparison.csv')

# Датасет с назначениями
df.to_csv('results/stations_with_clusters.csv', index=False)

# Веса признаков
weights_df = pd.DataFrame({
    'feature': config['data']['feature_cols'],
    'weight_baseline': baseline.weights / baseline.weights.sum(),
    'weight_weighted': weighted.weights / weighted.weights.sum(),
})
weights_df['weight_lift'] = weights_df['weight_weighted'] - weights_df['weight_baseline']
weights_df = weights_df.sort_values('weight_weighted', ascending=False)
weights_df.to_csv('results/feature_weights.csv', index=False)

print('\n=== ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ ===')
print(df_compare.to_string())
print('\n=== ВАЖНЕЙШИЕ ПРИЗНАКИ ===')
print(weights_df[['feature','weight_weighted','weight_lift']].head(6).to_string(index=False))
print('\n✓ Всё сохранено в results/')